In [2]:
# ================================================================
# COMPRESSIVE STRENGTH PREDICTOR
# NGBOOST REGRESSION APPLICATION
#
# Dataset:
# D:\2026 Work\My Papers\waste glass concrete\Data and Main Paper\
# Preprocessed\_Data\Final\_data_sushant_preprocessed.csv
#
# Target:
# CS
#
# NGBoost Hyperparameters:
# verbose_eval    = False
# verbose         = False
# tol             = 1.0e-05
# natural_gradient= True
# n_estimators    = 700
# minibatch_frac  = 0.7
# learning_rate   = 0.07
# col_sample      = 1.0
# random_state    = 42
# ================================================================

import os
import json
import time
import warnings
import threading
import joblib

import numpy as np
import pandas as pd

import tkinter as tk
from tkinter import ttk, messagebox, filedialog

from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from ngboost import NGBRegressor
from ngboost.distns import Normal


warnings.filterwarnings("ignore")


# ================================================================
# COMPRESSIVE STRENGTH PREDICTOR
# ================================================================

class CompressiveStrengthPredictor:

    # ============================================================
    # INITIALIZATION
    # ============================================================

    def __init__(self):

        self.setup_style()
        self.setup_paths()

        # --------------------------------------------------------
        # MODEL CONFIGURATION
        # --------------------------------------------------------

        self.TEST_SIZE = 0.20
        self.RANDOM_STATE = 42

        # ========================================================
        # FINAL NGBOOST HYPERPARAMETERS
        # ========================================================

        self.N_ESTIMATORS = 700
        self.LEARNING_RATE = 0.07
        self.MINIBATCH_FRAC = 0.7
        self.COL_SAMPLE = 1.0
        self.NATURAL_GRADIENT = True
        self.TOL = 1.0e-05

        self.model = None

        self.prediction_history = []

        # --------------------------------------------------------
        # DATA
        # --------------------------------------------------------

        self.load_and_preprocess_data()

        # --------------------------------------------------------
        # MODEL
        # --------------------------------------------------------

        self.load_or_train_model()

        # --------------------------------------------------------
        # GUI
        # --------------------------------------------------------

        self.create_gui()

        self.load_history()

    # ============================================================
    # STYLE
    # ============================================================

    def setup_style(self):

        self.colors = {

            "navy": "#17324D",
            "blue": "#2878B5",
            "light_blue": "#EAF3F9",

            "teal": "#1F8A8A",
            "light_teal": "#E8F6F6",

            "green": "#2E8B57",
            "light_green": "#EAF6EF",

            "orange": "#E67E22",
            "light_orange": "#FFF3E8",

            "red": "#C0392B",
            "light_red": "#FCECEA",

            "purple": "#6C5CE7",
            "light_purple": "#F0EEFF",

            "background": "#F4F7FA",
            "card": "#FFFFFF",
            "border": "#D9E1E8",

            "text": "#243447",
            "muted": "#6B7785",

            "white": "#FFFFFF"
        }

    # ============================================================
    # PATHS
    # ============================================================

    def setup_paths(self):

        # --------------------------------------------------------
        # DATA PATH
        # --------------------------------------------------------

        self.data_path = (
            r"D:\2026 Work\My Papers\waste glass concrete"
            r"\Data and Main Paper\Preprocessed\_Data\Final"
            r"\_data_sushant_preprocessed.csv"
        )

        # --------------------------------------------------------
        # RESULT DIRECTORY
        # --------------------------------------------------------

        self.save_dir = (
            r"D:\2026 Work\My Papers\waste glass concrete"
            r"\Data and Main Paper\Results\NGBoost_GUI"
        )

        os.makedirs(
            self.save_dir,
            exist_ok=True
        )

        # --------------------------------------------------------
        # MODEL
        # --------------------------------------------------------

        self.model_path = os.path.join(
            self.save_dir,
            "ngboost_CS_model.pkl"
        )

        # --------------------------------------------------------
        # PREPROCESSOR
        # --------------------------------------------------------

        self.preprocessor_path = os.path.join(
            self.save_dir,
            "CS_preprocessor.pkl"
        )

        # --------------------------------------------------------
        # METRICS
        # --------------------------------------------------------

        self.metrics_path = os.path.join(
            self.save_dir,
            "NGBoost_CS_metrics.json"
        )

        # --------------------------------------------------------
        # HISTORY
        # --------------------------------------------------------

        self.history_file = os.path.join(
            self.save_dir,
            "CS_prediction_history.json"
        )

    # ============================================================
    # DATA LOADING
    # ============================================================

    def load_and_preprocess_data(self):

        start = time.time()

        print("\n" + "=" * 75)
        print("LOADING COMPRESSIVE STRENGTH DATA")
        print("=" * 75)

        # --------------------------------------------------------
        # CHECK DATA PATH
        # --------------------------------------------------------

        if not os.path.exists(self.data_path):

            raise FileNotFoundError(
                "\nDataset not found.\n\n"
                f"Expected path:\n{self.data_path}\n"
            )

        # --------------------------------------------------------
        # READ CSV
        # --------------------------------------------------------

        encodings = [
            "utf-8",
            "utf-8-sig",
            "cp1252",
            "latin1"
        ]

        self.df = None

        for encoding in encodings:

            try:

                self.df = pd.read_csv(
                    self.data_path,
                    encoding=encoding
                )

                print(
                    f"CSV encoding: {encoding}"
                )

                break

            except UnicodeDecodeError:

                continue

        if self.df is None:

            raise ValueError(
                "Unable to decode the CSV file."
            )

        # --------------------------------------------------------
        # CLEAN COLUMN NAMES
        # --------------------------------------------------------

        self.df.columns = (
            self.df.columns
            .astype(str)
            .str.strip()
        )

        print(
            f"Dataset shape: {self.df.shape}"
        )

        print(
            "\nColumns:"
        )

        for i, column in enumerate(
            self.df.columns,
            start=1
        ):

            print(
                f"{i:3d}. {column}"
            )

        # ========================================================
        # TARGET
        # ========================================================

        self.target_name = "CS"

        if self.target_name not in self.df.columns:

            raise ValueError(
                "\nTarget column 'CS' was not found.\n\n"
                "Available columns:\n"
                +
                "\n".join(
                    str(c)
                    for c in self.df.columns
                )
            )

        print(
            f"\nTarget: {self.target_name}"
        )

        # --------------------------------------------------------
        # TARGET NUMERIC CONVERSION
        # --------------------------------------------------------

        self.df[self.target_name] = pd.to_numeric(
            self.df[self.target_name],
            errors="coerce"
        )

        # --------------------------------------------------------
        # REMOVE INVALID TARGETS
        # --------------------------------------------------------

        before_target_cleaning = len(self.df)

        self.df = self.df.dropna(
            subset=[self.target_name]
        )

        removed_target_rows = (
            before_target_cleaning -
            len(self.df)
        )

        if removed_target_rows > 0:

            print(
                f"Removed {removed_target_rows} "
                f"rows with missing CS."
            )

        # --------------------------------------------------------
        # REMOVE DUPLICATES
        # --------------------------------------------------------

        before_duplicates = len(self.df)

        self.df = (
            self.df
            .drop_duplicates()
            .reset_index(drop=True)
        )

        duplicate_rows = (
            before_duplicates -
            len(self.df)
        )

        if duplicate_rows > 0:

            print(
                f"Removed {duplicate_rows} duplicate rows."
            )

        # --------------------------------------------------------
        # REMOVE COMPLETELY EMPTY COLUMNS
        # --------------------------------------------------------

        empty_columns = [
            c
            for c in self.df.columns
            if self.df[c].isna().all()
        ]

        if empty_columns:

            print(
                "\nRemoving completely empty columns:"
            )

            for c in empty_columns:

                print(
                    f"  - {c}"
                )

            self.df = self.df.drop(
                columns=empty_columns
            )

        # ========================================================
        # X / Y
        # ========================================================

        self.X_full = (
            self.df
            .drop(
                columns=[self.target_name]
            )
            .copy()
        )

        self.y_full = (
            self.df[self.target_name]
            .copy()
        )

        self.feature_names = (
            self.X_full.columns.tolist()
        )

        # --------------------------------------------------------
        # TARGET SUMMARY
        # --------------------------------------------------------

        print("\nTarget statistics:")
        print(
            f"  n      = {len(self.y_full)}"
        )
        print(
            f"  min    = {self.y_full.min():.6f}"
        )
        print(
            f"  max    = {self.y_full.max():.6f}"
        )
        print(
            f"  mean   = {self.y_full.mean():.6f}"
        )
        print(
            f"  median = {self.y_full.median():.6f}"
        )
        print(
            f"  std    = {self.y_full.std():.6f}"
        )

        # ========================================================
        # IDENTIFY CATEGORICAL FEATURES
        # ========================================================

        self.categorical_features = []

        for feature in self.feature_names:

            dtype = self.X_full[
                feature
            ].dtype

            if (
                dtype == "object"
                or
                str(dtype) == "category"
            ):

                self.categorical_features.append(
                    feature
                )

        print(
            f"\nFeatures: "
            f"{len(self.feature_names)}"
        )

        print(
            f"Categorical features: "
            f"{len(self.categorical_features)}"
        )

        if self.categorical_features:

            for feature in self.categorical_features:

                print(
                    f"  - {feature}"
                )

        # ========================================================
        # TRAIN / TEST SPLIT
        # ========================================================

        (
            self.X_train_raw,
            self.X_test_raw,
            self.y_train,
            self.y_test
        ) = train_test_split(
            self.X_full,
            self.y_full,
            test_size=self.TEST_SIZE,
            random_state=self.RANDOM_STATE
        )

        print(
            "\nTrain/test split:"
        )

        print(
            f"  Training samples: "
            f"{len(self.X_train_raw)}"
        )

        print(
            f"  Test samples: "
            f"{len(self.X_test_raw)}"
        )

        # ========================================================
        # PREPROCESSING
        #
        # IMPORTANT:
        # All imputation/category mapping information is learned
        # ONLY from training data.
        # ========================================================

        self.feature_stats = {}
        self.categorical_maps = {}

        self.X_train = (
            self.X_train_raw.copy()
        )

        self.X_test = (
            self.X_test_raw.copy()
        )

        # --------------------------------------------------------
        # PROCESS EACH FEATURE
        # --------------------------------------------------------

        for feature in self.feature_names:

            # ====================================================
            # CATEGORICAL
            # ====================================================

            if feature in self.categorical_features:

                train_values = (
                    self.X_train[
                        feature
                    ]
                    .fillna("Missing")
                    .astype(str)
                )

                categories = sorted(
                    train_values.unique()
                )

                if "Missing" not in categories:

                    categories.append(
                        "Missing"
                    )

                category_to_code = {

                    category: index

                    for index, category
                    in enumerate(categories)

                }

                self.categorical_maps[
                    feature
                ] = category_to_code

                mode_values = (
                    train_values.mode()
                )

                if len(mode_values) > 0:

                    mode = (
                        mode_values.iloc[0]
                    )

                else:

                    mode = categories[0]

                self.feature_stats[
                    feature
                ] = {

                    "type": "categorical",

                    "categories":
                        categories,

                    "mode":
                        mode
                }

                # TRAIN

                self.X_train[
                    feature
                ] = (

                    train_values
                    .map(
                        category_to_code
                    )
                    .fillna(
                        category_to_code[
                            mode
                        ]
                    )
                    .astype(int)

                )

                # TEST

                test_values = (
                    self.X_test[
                        feature
                    ]
                    .fillna("Missing")
                    .astype(str)
                )

                self.X_test[
                    feature
                ] = (

                    test_values
                    .map(
                        category_to_code
                    )
                    .fillna(
                        category_to_code[
                            mode
                        ]
                    )
                    .astype(int)

                )

            # ====================================================
            # NUMERIC
            # ====================================================

            else:

                train_numeric = pd.to_numeric(
                    self.X_train[
                        feature
                    ],
                    errors="coerce"
                )

                train_numeric = (
                    train_numeric
                    .replace(
                        [np.inf, -np.inf],
                        np.nan
                    )
                )

                # ------------------------------------------------
                # TRAINING-DATA MEDIAN
                # ------------------------------------------------

                median_value = (
                    train_numeric.median()
                )

                mean_value = (
                    train_numeric.mean()
                )

                min_value = (
                    train_numeric.min()
                )

                max_value = (
                    train_numeric.max()
                )

                std_value = (
                    train_numeric.std()
                )

                # ------------------------------------------------
                # PROTECT AGAINST COMPLETELY INVALID FEATURE
                # ------------------------------------------------

                if pd.isna(median_value):

                    raise ValueError(
                        f"Feature '{feature}' "
                        "contains no valid numeric "
                        "training values."
                    )

                self.feature_stats[
                    feature
                ] = {

                    "type": "numeric",

                    "mean":
                        float(mean_value),

                    "median":
                        float(median_value),

                    "min":
                        float(min_value),

                    "max":
                        float(max_value),

                    "std":
                        float(std_value)
                        if np.isfinite(std_value)
                        else 0.0
                }

                # TRAIN

                self.X_train[
                    feature
                ] = (

                    train_numeric
                    .fillna(
                        median_value
                    )

                )

                # TEST

                test_numeric = pd.to_numeric(
                    self.X_test[
                        feature
                    ],
                    errors="coerce"
                )

                test_numeric = (
                    test_numeric
                    .replace(
                        [np.inf, -np.inf],
                        np.nan
                    )
                    .fillna(
                        median_value
                    )
                )

                self.X_test[
                    feature
                ] = test_numeric

        # ========================================================
        # FINAL NUMERIC VALIDATION
        # ========================================================

        if not np.isfinite(
            self.X_train
            .to_numpy(dtype=float)
        ).all():

            raise ValueError(
                "Non-finite values remain in "
                "training predictors."
            )

        if not np.isfinite(
            self.X_test
            .to_numpy(dtype=float)
        ).all():

            raise ValueError(
                "Non-finite values remain in "
                "test predictors."
            )

        # ========================================================
        # TARGET STATISTICS
        # ========================================================

        self.target_stats = {

            "min":
                float(self.y_train.min()),

            "max":
                float(self.y_train.max()),

            "mean":
                float(self.y_train.mean()),

            "std":
                float(self.y_train.std()),

            "median":
                float(self.y_train.median())
        }

        # ========================================================
        # SAVE PREPROCESSOR
        # ========================================================

        preprocessor = {

            "feature_names":
                self.feature_names,

            "categorical_features":
                self.categorical_features,

            "categorical_maps":
                self.categorical_maps,

            "feature_stats":
                self.feature_stats,

            "target_stats":
                self.target_stats,

            "target_name":
                self.target_name,

            "random_state":
                self.RANDOM_STATE,

            "test_size":
                self.TEST_SIZE
        }

        joblib.dump(
            preprocessor,
            self.preprocessor_path
        )

        print(
            "\nPreprocessor saved:"
        )

        print(
            self.preprocessor_path
        )

        print(
            f"\nPreprocessing completed in "
            f"{time.time() - start:.2f} s"
        )

    # ============================================================
    # MODEL
    # ============================================================

    def load_or_train_model(self):

        start = time.time()

        # ========================================================
        # LOAD EXISTING MODEL
        # ========================================================

        if os.path.exists(
            self.model_path
        ):

            try:

                print(
                    "\n" + "=" * 75
                )

                print(
                    "LOADING SAVED NGBOOST MODEL"
                )

                print(
                    "=" * 75
                )

                self.model = joblib.load(
                    self.model_path
                )

                print(
                    "\nModel loaded:"
                )

                print(
                    type(self.model)
                )

                self.calculate_test_metrics()

                print(
                    "\nSaved NGBoost model loaded successfully."
                )

                return

            except Exception as e:

                print(
                    "\nSaved model loading failed:"
                )

                print(
                    str(e)
                )

                print(
                    "\nA new model will be trained."
                )

                self.model = None

        # ========================================================
        # TRAIN NGBOOST
        # ========================================================

        print(
            "\n" + "=" * 75
        )

        print(
            "TRAINING FINAL NGBOOST MODEL"
        )

        print(
            "=" * 75
        )

        print(
            "\nHyperparameters:"
        )

        print(
            f"  n_estimators     = "
            f"{self.N_ESTIMATORS}"
        )

        print(
            f"  learning_rate    = "
            f"{self.LEARNING_RATE}"
        )

        print(
            f"  minibatch_frac   = "
            f"{self.MINIBATCH_FRAC}"
        )

        print(
            f"  col_sample       = "
            f"{self.COL_SAMPLE}"
        )

        print(
            f"  natural_gradient = "
            f"{self.NATURAL_GRADIENT}"
        )

        print(
            f"  tol              = "
            f"{self.TOL}"
        )

        print(
            f"  random_state     = "
            f"{self.RANDOM_STATE}"
        )

        print(
            f"  verbose          = False"
        )

        print(
            f"  verbose_eval     = False"
        )

        # ========================================================
        # NGBOOST
        # ========================================================

        self.model = NGBRegressor(

            Dist=Normal,

            n_estimators=
                self.N_ESTIMATORS,

            learning_rate=
                self.LEARNING_RATE,

            minibatch_frac=
                self.MINIBATCH_FRAC,

            col_sample=
                self.COL_SAMPLE,

            natural_gradient=
                self.NATURAL_GRADIENT,

            tol=
                self.TOL,

            random_state=
                self.RANDOM_STATE,

            verbose=
                False,

            verbose_eval=
                False
        )

        # ========================================================
        # TRAIN
        # ========================================================

        train_start = time.time()

        self.model.fit(
            self.X_train,
            self.y_train
        )

        train_time = (
            time.time() -
            train_start
        )

        print(
            f"\nTraining time: "
            f"{train_time:.2f} seconds"
        )

        # ========================================================
        # TEST PERFORMANCE
        # ========================================================

        self.calculate_test_metrics()

        # ========================================================
        # SAVE MODEL
        # ========================================================

        joblib.dump(
            self.model,
            self.model_path
        )

        print(
            "\nNGBoost model saved:"
        )

        print(
            self.model_path
        )

        print(
            f"\nInitialization time: "
            f"{time.time() - start:.2f} seconds"
        )

    # ============================================================
    # TEST METRICS
    # ============================================================

    def calculate_test_metrics(self):

        # --------------------------------------------------------
        # PREDICTIONS
        # --------------------------------------------------------

        prediction_result = (
            self.model.predict(
                self.X_test
            )
        )

        self.y_test_pred = np.asarray(
            prediction_result,
            dtype=float
        )

        # --------------------------------------------------------
        # METRICS
        # --------------------------------------------------------

        self.test_r2 = r2_score(
            self.y_test,
            self.y_test_pred
        )

        self.test_rmse = np.sqrt(
            mean_squared_error(
                self.y_test,
                self.y_test_pred
            )
        )

        self.test_mae = (
            mean_absolute_error(
                self.y_test,
                self.y_test_pred
            )
        )

        # --------------------------------------------------------
        # MAPE
        #
        # MAPE is reported only for non-zero actual values.
        # --------------------------------------------------------

        y_true = (
            self.y_test.to_numpy(
                dtype=float
            )
        )

        y_pred = (
            self.y_test_pred
        )

        nonzero = (
            np.abs(y_true) > 1e-12
        )

        if np.any(nonzero):

            self.test_mape = (

                np.mean(

                    np.abs(

                        (
                            y_true[nonzero]
                            -
                            y_pred[nonzero]
                        )
                        /
                        y_true[nonzero]

                    )

                )
                * 100

            )

        else:

            self.test_mape = np.nan

        # ========================================================
        # ADDITIONAL METRICS
        # ========================================================

        residuals = (
            y_true -
            y_pred
        )

        self.residual_mean = (
            float(np.mean(residuals))
        )

        self.residual_std = (
            float(np.std(residuals))
        )

        # ========================================================
        # SAVE METRICS
        # ========================================================

        metrics = {

            "model":
                "NGBoost NGBRegressor",

            "distribution":
                "Normal",

            "target":
                self.target_name,

            "n_estimators":
                self.N_ESTIMATORS,

            "learning_rate":
                self.LEARNING_RATE,

            "minibatch_frac":
                self.MINIBATCH_FRAC,

            "col_sample":
                self.COL_SAMPLE,

            "natural_gradient":
                self.NATURAL_GRADIENT,

            "tol":
                self.TOL,

            "random_state":
                self.RANDOM_STATE,

            "test_size":
                self.TEST_SIZE,

            "training_samples":
                len(self.X_train),

            "test_samples":
                len(self.X_test),

            "features":
                len(self.feature_names),

            "test_R2":
                float(self.test_r2),

            "test_RMSE":
                float(self.test_rmse),

            "test_MAE":
                float(self.test_mae),

            "test_MAPE_percent":
                (
                    float(self.test_mape)
                    if np.isfinite(
                        self.test_mape
                    )
                    else None
                ),

            "residual_mean":
                self.residual_mean,

            "residual_std":
                self.residual_std
        }

        with open(
            self.metrics_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                metrics,
                f,
                indent=4
            )

        # ========================================================
        # PRINT RESULTS
        # ========================================================

        print(
            "\n" + "=" * 75
        )

        print(
            "FINAL TEST-SET PERFORMANCE"
        )

        print(
            "=" * 75
        )

        print(
            f"R²   = {self.test_r2:.6f}"
        )

        print(
            f"RMSE = {self.test_rmse:.6f}"
        )

        print(
            f"MAE  = {self.test_mae:.6f}"
        )

        if np.isfinite(
            self.test_mape
        ):

            print(
                f"MAPE = "
                f"{self.test_mape:.4f}%"
            )

        print(
            "=" * 75
        )

    # ============================================================
    # GUI
    # ============================================================

    def create_gui(self):

        self.root = tk.Tk()

        self.root.title(
            "Compressive Strength Predictor | NGBoost"
        )

        self.root.geometry(
            "1450x900"
        )

        self.root.minsize(
            1150,
            750
        )

        self.root.configure(
            bg=self.colors["background"]
        )

        self.configure_ttk()

        # ========================================================
        # HEADER
        # ========================================================

        header = tk.Frame(
            self.root,
            bg=self.colors["navy"],
            height=90
        )

        header.pack(
            fill="x"
        )

        header.pack_propagate(
            False
        )

        title_frame = tk.Frame(
            header,
            bg=self.colors["navy"]
        )

        title_frame.pack(
            side="left",
            padx=28,
            pady=12
        )

        tk.Label(
            title_frame,
            text="COMPRESSIVE STRENGTH PREDICTOR",
            font=("Segoe UI", 21, "bold"),
            fg=self.colors["white"],
            bg=self.colors["navy"]
        ).pack(
            anchor="w"
        )

        tk.Label(
            title_frame,
            text="NGBoost Regression Application",
            font=("Segoe UI", 10),
            fg="#D5E5F2",
            bg=self.colors["navy"]
        ).pack(
            anchor="w",
            pady=(3, 0)
        )

        model_badge = tk.Label(
            header,
            text="NGBOOST",
            font=("Segoe UI", 10, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["white"],
            padx=15,
            pady=7
        )

        model_badge.pack(
            side="right",
            padx=28
        )

        # ========================================================
        # NOTEBOOK
        # ========================================================

        self.notebook = ttk.Notebook(
            self.root
        )

        self.notebook.pack(
            fill="both",
            expand=True,
            padx=14,
            pady=12
        )

        self.prediction_tab = (
            self.create_prediction_tab()
        )

        self.analysis_tab = (
            self.create_analysis_tab()
        )

        self.history_tab = (
            self.create_history_tab()
        )

        self.notebook.add(
            self.prediction_tab,
            text="  Prediction  "
        )

        self.notebook.add(
            self.analysis_tab,
            text="  Analysis  "
        )

        self.notebook.add(
            self.history_tab,
            text="  History  "
        )

        # ========================================================
        # STATUS BAR
        # ========================================================

        status_frame = tk.Frame(
            self.root,
            bg=self.colors["navy"],
            height=32
        )

        status_frame.pack(
            fill="x"
        )

        status_frame.pack_propagate(
            False
        )

        self.status_var = tk.StringVar(
            value="Model ready"
        )

        tk.Label(
            status_frame,
            textvariable=self.status_var,
            font=("Segoe UI", 9),
            fg=self.colors["white"],
            bg=self.colors["navy"]
        ).pack(
            side="left",
            padx=15
        )

        self.root.protocol(
            "WM_DELETE_WINDOW",
            self.on_closing
        )

    # ============================================================
    # TTK STYLE
    # ============================================================

    def configure_ttk(self):

        style = ttk.Style()

        try:

            style.theme_use(
                "clam"
            )

        except Exception:

            pass

        style.configure(
            ".",
            font=("Segoe UI", 10)
        )

        style.configure(
            "TNotebook",
            background=self.colors["background"],
            borderwidth=0
        )

        style.configure(
            "TNotebook.Tab",
            padding=(20, 10),
            font=("Segoe UI", 10, "bold")
        )

        style.configure(
            "TFrame",
            background=self.colors["background"]
        )

        style.configure(
            "TLabelframe",
            background=self.colors["background"]
        )

        style.configure(
            "TLabelframe.Label",
            font=("Segoe UI", 10, "bold"),
            foreground=self.colors["navy"]
        )

        style.configure(
            "TButton",
            font=("Segoe UI", 10, "bold"),
            padding=(12, 8)
        )

        style.configure(
            "Accent.TButton",
            foreground=self.colors["white"],
            background=self.colors["blue"],
            font=("Segoe UI", 11, "bold"),
            padding=(18, 10)
        )

        style.map(
            "Accent.TButton",
            background=[
                (
                    "active",
                    self.colors["navy"]
                )
            ]
        )

        style.configure(
            "TEntry",
            padding=7
        )

        style.configure(
            "TCombobox",
            padding=6
        )

    # ============================================================
    # PREDICTION TAB
    # ============================================================

    def create_prediction_tab(self):

        tab = ttk.Frame(
            self.notebook
        )

        main = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        main.pack(
            fill="both",
            expand=True,
            padx=12,
            pady=12
        )

        # ========================================================
        # LEFT PANEL
        # ========================================================

        left_card = tk.Frame(
            main,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        left_card.pack(
            side="left",
            fill="both",
            expand=True,
            padx=(0, 8)
        )

        tk.Label(
            left_card,
            text="Input Parameters",
            font=("Segoe UI", 15, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["card"]
        ).pack(
            anchor="w",
            padx=20,
            pady=(18, 2)
        )

        tk.Label(
            left_card,
            text="Enter the concrete mix parameters.",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["card"]
        ).pack(
            anchor="w",
            padx=20,
            pady=(0, 12)
        )

        # ========================================================
        # SCROLL AREA
        # ========================================================

        canvas = tk.Canvas(
            left_card,
            bg=self.colors["card"],
            highlightthickness=0
        )

        scrollbar = ttk.Scrollbar(
            left_card,
            orient="vertical",
            command=canvas.yview
        )

        input_frame = tk.Frame(
            canvas,
            bg=self.colors["card"]
        )

        input_frame.bind(
            "<Configure>",
            lambda e:
            canvas.configure(
                scrollregion=
                canvas.bbox("all")
            )
        )

        canvas_window = canvas.create_window(
            (0, 0),
            window=input_frame,
            anchor="nw"
        )

        canvas.bind(
            "<Configure>",
            lambda e:
            canvas.itemconfig(
                canvas_window,
                width=e.width
            )
        )

        canvas.configure(
            yscrollcommand=
            scrollbar.set
        )

        canvas.pack(
            side="left",
            fill="both",
            expand=True,
            padx=(15, 0)
        )

        scrollbar.pack(
            side="right",
            fill="y",
            pady=5
        )

        # ========================================================
        # QUICK INPUT
        # ========================================================

        quick = tk.Frame(
            input_frame,
            bg=self.colors["light_blue"],
            highlightbackground="#C9DFEF",
            highlightthickness=1
        )

        quick.pack(
            fill="x",
            padx=5,
            pady=(0, 12)
        )

        tk.Label(
            quick,
            text="Quick samples",
            font=("Segoe UI", 9, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["light_blue"]
        ).pack(
            side="left",
            padx=10,
            pady=8
        )

        ttk.Button(
            quick,
            text="Random",
            command=self.fill_random_sample
        ).pack(
            side="left",
            padx=3,
            pady=5
        )

        ttk.Button(
            quick,
            text="Minimum CS",
            command=self.fill_best_case
        ).pack(
            side="left",
            padx=3,
            pady=5
        )

        ttk.Button(
            quick,
            text="Maximum CS",
            command=self.fill_worst_case
        ).pack(
            side="left",
            padx=3,
            pady=5
        )

        # ========================================================
        # INPUTS
        # ========================================================

        self.entries = {}

        for feature in self.feature_names:

            row = tk.Frame(
                input_frame,
                bg=self.colors["card"]
            )

            row.pack(
                fill="x",
                padx=5,
                pady=4
            )

            tk.Label(
                row,
                text=feature,
                font=("Segoe UI", 9, "bold"),
                fg=self.colors["text"],
                bg=self.colors["card"],
                anchor="w",
                width=30
            ).pack(
                side="left"
            )

            stats = (
                self.feature_stats[
                    feature
                ]
            )

            if feature in self.categorical_features:

                combo = ttk.Combobox(
                    row,
                    width=24,
                    state="readonly"
                )

                categories = (
                    stats["categories"]
                )

                combo["values"] = (
                    categories
                )

                combo.set(
                    stats["mode"]
                )

                combo.pack(
                    side="left",
                    padx=6
                )

                self.entries[
                    feature
                ] = combo

            else:

                entry = ttk.Entry(
                    row,
                    width=18
                )

                entry.insert(
                    0,
                    f"{stats['mean']:.4f}"
                )

                entry.pack(
                    side="left",
                    padx=6
                )

                tk.Label(
                    row,
                    text=(
                        f"{stats['min']:.3f}  to  "
                        f"{stats['max']:.3f}"
                    ),
                    font=("Segoe UI", 8),
                    fg=self.colors["muted"],
                    bg=self.colors["card"]
                ).pack(
                    side="left"
                )

                self.entries[
                    feature
                ] = entry

        # ========================================================
        # BUTTONS
        # ========================================================

        button_frame = tk.Frame(
            input_frame,
            bg=self.colors["card"]
        )

        button_frame.pack(
            fill="x",
            padx=5,
            pady=18
        )

        self.predict_btn = ttk.Button(
            button_frame,
            text="PREDICT CS",
            style="Accent.TButton",
            command=self.threaded_predict
        )

        self.predict_btn.pack(
            side="left",
            padx=3
        )

        ttk.Button(
            button_frame,
            text="Reset",
            command=self.clear_inputs
        ).pack(
            side="left",
            padx=5
        )

        # ========================================================
        # RIGHT PANEL
        # ========================================================

        right_card = tk.Frame(
            main,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        right_card.pack(
            side="right",
            fill="both",
            expand=True,
            padx=(8, 0)
        )

        tk.Label(
            right_card,
            text="Prediction Result",
            font=("Segoe UI", 15, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["card"]
        ).pack(
            anchor="w",
            padx=22,
            pady=(18, 0)
        )

        tk.Label(
            right_card,
            text="Estimated compressive strength",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["card"]
        ).pack(
            anchor="w",
            padx=22
        )

        # ========================================================
        # RESULT CARD
        # ========================================================

        result_card = tk.Frame(
            right_card,
            bg=self.colors["light_blue"],
            highlightbackground="#C9DFEF",
            highlightthickness=1
        )

        result_card.pack(
            fill="x",
            padx=22,
            pady=18
        )

        self.result_var = tk.StringVar(
            value="READY"
        )

        tk.Label(
            result_card,
            textvariable=self.result_var,
            font=("Segoe UI", 34, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["light_blue"]
        ).pack(
            pady=(25, 0)
        )

        tk.Label(
            result_card,
            text="CS",
            font=("Segoe UI", 11),
            fg=self.colors["muted"],
            bg=self.colors["light_blue"]
        ).pack(
            pady=(0, 25)
        )

        # ========================================================
        # PREDICTION INTERVAL
        # ========================================================

        self.interval_var = tk.StringVar(
            value=""
        )

        tk.Label(
            result_card,
            textvariable=self.interval_var,
            font=("Segoe UI", 10),
            fg=self.colors["blue"],
            bg=self.colors["light_blue"]
        ).pack(
            pady=(0, 18)
        )

        # ========================================================
        # ACTIONS
        # ========================================================

        actions = tk.Frame(
            right_card,
            bg=self.colors["card"]
        )

        actions.pack(
            fill="x",
            padx=22,
            pady=5
        )

        ttk.Button(
            actions,
            text="Save Result",
            command=self.save_result
        ).pack(
            side="left",
            padx=(0, 5)
        )

        ttk.Button(
            actions,
            text="Copy",
            command=self.copy_to_clipboard
        ).pack(
            side="left",
            padx=5
        )

        ttk.Button(
            actions,
            text="Export History",
            command=self.export_history
        ).pack(
            side="left",
            padx=5
        )

        return tab

    # ============================================================
    # ANALYSIS TAB
    # ============================================================

    def create_analysis_tab(self):

        tab = ttk.Frame(
            self.notebook
        )

        top = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        top.pack(
            fill="x",
            padx=15,
            pady=(15, 8)
        )

        tk.Label(
            top,
            text="Model Analysis",
            font=("Segoe UI", 17, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["background"]
        ).pack(
            side="left"
        )

        tk.Label(
            top,
            text="Independent test-set diagnostics",
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["background"]
        ).pack(
            side="left",
            padx=15,
            pady=5
        )

        # ========================================================
        # METRICS
        # ========================================================

        metric_frame = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        metric_frame.pack(
            fill="x",
            padx=15,
            pady=5
        )

        metrics = [

            (
                "R²",
                f"{self.test_r2:.4f}",
                self.colors["light_blue"],
                self.colors["navy"]
            ),

            (
                "RMSE",
                f"{self.test_rmse:.4f}",
                self.colors["light_teal"],
                self.colors["teal"]
            ),

            (
                "MAE",
                f"{self.test_mae:.4f}",
                self.colors["light_green"],
                self.colors["green"]
            ),

            (
                "MAPE",
                (
                    f"{self.test_mape:.2f}%"
                    if np.isfinite(
                        self.test_mape
                    )
                    else "N/A"
                ),
                self.colors["light_orange"],
                self.colors["orange"]
            )
        ]

        for title, value, bg, fg in metrics:

            card = self.metric_card(
                metric_frame,
                title,
                value,
                bg,
                fg
            )

            card.pack(
                side="left",
                fill="x",
                expand=True,
                padx=5
            )

        # ========================================================
        # BUTTONS
        # ========================================================

        controls = tk.Frame(
            tab,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        controls.pack(
            fill="x",
            padx=15,
            pady=5
        )

        buttons = [

            (
                "Actual vs Predicted",
                self.plot_actual_vs_predicted
            ),

            (
                "Residual Analysis",
                self.plot_residual_analysis
            ),

            (
                "Residual Histogram",
                self.plot_residual_histogram
            ),

            (
                "Correlation Matrix",
                self.plot_correlation_matrix
            )
        ]

        for text, command in buttons:

            ttk.Button(
                controls,
                text=text,
                command=command,
                width=22
            ).pack(
                side="left",
                padx=5,
                pady=10
            )

        self.analysis_frame = tk.Frame(
            tab,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        self.analysis_frame.pack(
            fill="both",
            expand=True,
            padx=15,
            pady=10
        )

        self.plot_actual_vs_predicted()

        return tab

    # ============================================================
    # METRIC CARD
    # ============================================================

    def metric_card(
        self,
        parent,
        title,
        value,
        background,
        foreground
    ):

        card = tk.Frame(
            parent,
            bg=background,
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        tk.Label(
            card,
            text=title,
            font=("Segoe UI", 9, "bold"),
            fg=self.colors["muted"],
            bg=background
        ).pack(
            pady=(12, 0)
        )

        tk.Label(
            card,
            text=value,
            font=("Segoe UI", 17, "bold"),
            fg=foreground,
            bg=background
        ).pack(
            pady=(2, 12)
        )

        return card

    # ============================================================
    # HISTORY TAB
    # ============================================================

    def create_history_tab(self):

        tab = ttk.Frame(
            self.notebook
        )

        top = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        top.pack(
            fill="x",
            padx=15,
            pady=15
        )

        tk.Label(
            top,
            text="Prediction History",
            font=("Segoe UI", 17, "bold"),
            fg=self.colors["navy"],
            bg=self.colors["background"]
        ).pack(
            side="left"
        )

        self.history_stats_var = (
            tk.StringVar(
                value="No predictions yet"
            )
        )

        tk.Label(
            top,
            textvariable=self.history_stats_var,
            font=("Segoe UI", 9),
            fg=self.colors["muted"],
            bg=self.colors["background"]
        ).pack(
            side="right"
        )

        controls = tk.Frame(
            tab,
            bg=self.colors["background"]
        )

        controls.pack(
            fill="x",
            padx=15
        )

        ttk.Button(
            controls,
            text="Refresh",
            command=self.update_history_display
        ).pack(
            side="left",
            padx=4
        )

        ttk.Button(
            controls,
            text="Export CSV / Excel",
            command=self.export_history
        ).pack(
            side="left",
            padx=4
        )

        ttk.Button(
            controls,
            text="Clear History",
            command=self.clear_history
        ).pack(
            side="left",
            padx=4
        )

        # ========================================================
        # LIST
        # ========================================================

        card = tk.Frame(
            tab,
            bg=self.colors["card"],
            highlightbackground=self.colors["border"],
            highlightthickness=1
        )

        card.pack(
            fill="both",
            expand=True,
            padx=15,
            pady=12
        )

        self.history_listbox = tk.Listbox(
            card,
            font=("Consolas", 10),
            bg=self.colors["card"],
            fg=self.colors["text"],
            selectbackground=self.colors["blue"],
            selectforeground=self.colors["white"],
            relief="flat",
            borderwidth=0
        )

        self.history_listbox.pack(
            side="left",
            fill="both",
            expand=True,
            padx=10,
            pady=10
        )

        scrollbar = ttk.Scrollbar(
            card,
            orient="vertical",
            command=self.history_listbox.yview
        )

        scrollbar.pack(
            side="right",
            fill="y",
            pady=10
        )

        self.history_listbox.configure(
            yscrollcommand=scrollbar.set
        )

        self.history_listbox.bind(
            "<Double-Button-1>",
            self.load_history_entry
        )

        return tab

    # ============================================================
    # INPUT HELPERS
    # ============================================================

    def fill_random_sample(self):

        if len(self.X_train_raw) == 0:

            return

        index = np.random.randint(
            len(self.X_train_raw)
        )

        sample = (
            self.X_train_raw.iloc[
                index
            ]
        )

        self.fill_raw_sample(
            sample
        )

    # ============================================================

    def fill_raw_sample(
        self,
        sample
    ):

        for feature in self.feature_names:

            if feature not in sample:

                continue

            widget = self.entries[
                feature
            ]

            value = sample[
                feature
            ]

            if feature in self.categorical_features:

                widget.set(
                    str(value)
                )

            else:

                widget.delete(
                    0,
                    tk.END
                )

                widget.insert(
                    0,
                    str(value)
                )

    # ============================================================

    def fill_best_case(self):

        idx = self.y_train.idxmin()

        sample = (
            self.X_train_raw.loc[
                idx
            ]
        )

        self.fill_raw_sample(
            sample
        )

    # ============================================================

    def fill_worst_case(self):

        idx = self.y_train.idxmax()

        sample = (
            self.X_train_raw.loc[
                idx
            ]
        )

        self.fill_raw_sample(
            sample
        )

    # ============================================================
    # BUILD PREDICTION DATA
    # ============================================================

    def build_prediction_dataframe(self):

        data = {}

        for feature in self.feature_names:

            widget = self.entries[
                feature
            ]

            value = widget.get().strip()

            # ====================================================
            # CATEGORICAL
            # ====================================================

            if feature in self.categorical_features:

                mapping = (
                    self.categorical_maps[
                        feature
                    ]
                )

                if value not in mapping:

                    raise ValueError(
                        f"Unknown category for "
                        f"'{feature}': {value}"
                    )

                data[feature] = (
                    mapping[value]
                )

            # ====================================================
            # NUMERIC
            # ====================================================

            else:

                try:

                    numeric_value = float(
                        value
                    )

                except ValueError:

                    raise ValueError(
                        f"Invalid numeric value "
                        f"for '{feature}'."
                    )

                if not np.isfinite(
                    numeric_value
                ):

                    raise ValueError(
                        f"Invalid value for "
                        f"'{feature}'."
                    )

                data[feature] = (
                    numeric_value
                )

        return pd.DataFrame(
            [data],
            columns=self.feature_names
        )

    # ============================================================
    # PREDICTION
    # ============================================================

    def threaded_predict(self):

        self.predict_btn.config(
            state="disabled",
            text="PREDICTING..."
        )

        self.status_var.set(
            "Generating NGBoost prediction..."
        )

        thread = threading.Thread(
            target=self.predict_strength,
            daemon=True
        )

        thread.start()

    # ============================================================

    def predict_strength(self):

        try:

            input_data = (
                self.build_prediction_dataframe()
            )

            start = time.time()

            # ----------------------------------------------------
            # MEAN PREDICTION
            # ----------------------------------------------------

            prediction = float(
                self.model.predict(
                    input_data
                )[0]
            )

            # ----------------------------------------------------
            # NGBOOST PREDICTIVE DISTRIBUTION
            #
            # For NGBRegressor with Normal distribution,
            # pred_dist() provides the fitted distribution.
            # ----------------------------------------------------

            lower = None
            upper = None

            try:

                pred_dist = (
                    self.model.pred_dist(
                        input_data
                    )
                )

                # NGBoost Normal distribution
                # exposes loc and scale.

                loc = np.asarray(
                    pred_dist.loc
                ).reshape(-1)

                scale = np.asarray(
                    pred_dist.scale
                ).reshape(-1)

                if (
                    len(loc) > 0
                    and
                    len(scale) > 0
                    and
                    np.isfinite(
                        scale[0]
                    )
                ):

                    lower = float(
                        loc[0]
                        -
                        1.96 * scale[0]
                    )

                    upper = float(
                        loc[0]
                        +
                        1.96 * scale[0]
                    )

            except Exception:

                # Prediction still remains valid
                # if distribution extraction is unavailable.

                lower = None
                upper = None

            elapsed = (
                time.time() -
                start
            )

            self.root.after(
                0,
                lambda:
                self.display_result(
                    prediction,
                    input_data,
                    elapsed,
                    lower,
                    upper
                )
            )

        except Exception as e:

            self.root.after(
                0,
                lambda:
                messagebox.showerror(
                    "Prediction Error",
                    str(e)
                )
            )

        finally:

            self.root.after(
                0,
                self.enable_predict_button
            )

    # ============================================================
    # ENABLE BUTTON
    # ============================================================

    def enable_predict_button(self):

        self.predict_btn.config(
            state="normal",
            text="PREDICT CS"
        )

        self.status_var.set(
            "Model ready"
        )

    # ============================================================
    # DISPLAY RESULT
    # ============================================================

    def display_result(
        self,
        prediction,
        input_data,
        elapsed,
        lower=None,
        upper=None
    ):

        self.result_var.set(
            f"{prediction:.4f}"
        )

        # --------------------------------------------------------
        # PREDICTION INTERVAL
        # --------------------------------------------------------

        if (
            lower is not None
            and
            upper is not None
        ):

            self.interval_var.set(
                f"Approx. 95% predictive interval: "
                f"{lower:.4f} to {upper:.4f}"
            )

        else:

            self.interval_var.set(
                ""
            )

        self.status_var.set(
            f"Prediction completed in "
            f"{elapsed:.3f} s"
        )

        # --------------------------------------------------------
        # SAVE HISTORY
        # --------------------------------------------------------

        self.save_to_history(
            prediction,
            lower,
            upper,
            input_data
        )

    # ============================================================
    # HISTORY
    # ============================================================

    def save_to_history(
        self,
        prediction,
        lower,
        upper,
        input_data
    ):

        entry = {

            "timestamp":
                time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),

            "CS":
                float(prediction),

            "lower_95":
                (
                    float(lower)
                    if lower is not None
                    else None
                ),

            "upper_95":
                (
                    float(upper)
                    if upper is not None
                    else None
                )
        }

        for feature in self.feature_names:

            value = input_data[
                feature
            ].iloc[0]

            entry[feature] = (

                float(value)
                if np.isscalar(value)
                else str(value)

            )

        self.prediction_history.append(
            entry
        )

        self.save_history()

        self.update_history_display()

    # ============================================================

    def update_history_display(self):

        self.history_listbox.delete(
            0,
            tk.END
        )

        for entry in self.prediction_history:

            cs_value = entry.get(
                "CS",
                np.nan
            )

            text = (

                f"{entry['timestamp']}"
                f"   |   "
                f"CS = {cs_value:.4f}"

            )

            lower = entry.get(
                "lower_95"
            )

            upper = entry.get(
                "upper_95"
            )

            if (
                lower is not None
                and
                upper is not None
            ):

                text += (
                    f"   |   "
                    f"95% PI = "
                    f"{lower:.3f}–{upper:.3f}"
                )

            self.history_listbox.insert(
                tk.END,
                text
            )

        # ========================================================
        # SUMMARY
        # ========================================================

        if self.prediction_history:

            values = [

                x["CS"]

                for x
                in self.prediction_history

                if "CS" in x

            ]

            if values:

                self.history_stats_var.set(

                    f"{len(values)} predictions"
                    f"   |   "
                    f"Mean {np.mean(values):.4f}"
                    f"   |   "
                    f"Min {np.min(values):.4f}"
                    f"   |   "
                    f"Max {np.max(values):.4f}"

                )

        else:

            self.history_stats_var.set(
                "No predictions yet"
            )

    # ============================================================

    def load_history_entry(
        self,
        event=None
    ):

        selection = (
            self.history_listbox
            .curselection()
        )

        if not selection:

            return

        index = selection[0]

        if index >= len(
            self.prediction_history
        ):

            return

        entry = (
            self.prediction_history[
                index
            ]
        )

        # --------------------------------------------------------
        # LOAD FEATURES
        # --------------------------------------------------------

        for feature in self.feature_names:

            if feature not in entry:

                continue

            widget = self.entries[
                feature
            ]

            value = str(
                entry[feature]
            )

            if feature in self.categorical_features:

                widget.set(
                    value
                )

            else:

                widget.delete(
                    0,
                    tk.END
                )

                widget.insert(
                    0,
                    value
                )

        # --------------------------------------------------------
        # LOAD RESULT
        # --------------------------------------------------------

        if "CS" in entry:

            self.result_var.set(
                f"{entry['CS']:.4f}"
            )

        lower = entry.get(
            "lower_95"
        )

        upper = entry.get(
            "upper_95"
        )

        if (
            lower is not None
            and
            upper is not None
        ):

            self.interval_var.set(
                f"Approx. 95% predictive interval: "
                f"{lower:.4f} to {upper:.4f}"
            )

        else:

            self.interval_var.set(
                ""
            )

    # ============================================================

    def save_history(self):

        try:

            with open(
                self.history_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    self.prediction_history,
                    f,
                    indent=4
                )

        except Exception as e:

            print(
                f"History save error: {e}"
            )

    # ============================================================

    def load_history(self):

        if not os.path.exists(
            self.history_file
        ):

            return

        try:

            with open(
                self.history_file,
                "r",
                encoding="utf-8"
            ) as f:

                self.prediction_history = (
                    json.load(f)
                )

            self.update_history_display()

        except Exception:

            self.prediction_history = []

    # ============================================================

    def clear_history(self):

        if not self.prediction_history:

            return

        answer = messagebox.askyesno(
            "Clear History",
            "Delete all prediction history?"
        )

        if answer:

            self.prediction_history = []

            self.save_history()

            self.update_history_display()

    # ============================================================

    def export_history(self):

        if not self.prediction_history:

            messagebox.showinfo(
                "No Data",
                "No prediction history exists."
            )

            return

        filename = (
            filedialog.asksaveasfilename(
                defaultextension=".csv",
                filetypes=[
                    (
                        "CSV files",
                        "*.csv"
                    ),
                    (
                        "Excel files",
                        "*.xlsx"
                    )
                ]
            )
        )

        if not filename:

            return

        df = pd.DataFrame(
            self.prediction_history
        )

        if filename.lower().endswith(
            ".xlsx"
        ):

            df.to_excel(
                filename,
                index=False
            )

        else:

            df.to_csv(
                filename,
                index=False
            )

        messagebox.showinfo(
            "Export Complete",
            "Prediction history exported successfully."
        )

    # ============================================================
    # CLEAR INPUTS
    # ============================================================

    def clear_inputs(self):

        for feature in self.feature_names:

            widget = self.entries[
                feature
            ]

            stats = (
                self.feature_stats[
                    feature
                ]
            )

            if feature in self.categorical_features:

                widget.set(
                    stats["mode"]
                )

            else:

                widget.delete(
                    0,
                    tk.END
                )

                widget.insert(
                    0,
                    f"{stats['mean']:.4f}"
                )

        self.result_var.set(
            "READY"
        )

        self.interval_var.set(
            ""
        )

        self.status_var.set(
            "Model ready"
        )

    # ============================================================
    # SAVE RESULT
    # ============================================================

    def save_result(self):

        if self.result_var.get() == "READY":

            messagebox.showinfo(
                "No Prediction",
                "Generate a prediction first."
            )

            return

        filename = (
            filedialog.asksaveasfilename(
                defaultextension=".txt",
                filetypes=[
                    (
                        "Text files",
                        "*.txt"
                    )
                ],
                initialfile=
                "CS_prediction.txt"
            )
        )

        if not filename:

            return

        text = (

            "COMPRESSIVE STRENGTH PREDICTION\n"
            "========================================\n\n"

            f"Predicted CS: "
            f"{self.result_var.get()}\n\n"

            f"{self.interval_var.get()}\n\n"

            "INPUT PARAMETERS\n"
            "----------------------------------------\n"

        )

        for feature in self.feature_names:

            text += (

                f"{feature}: "
                f"{self.entries[feature].get()}\n"

            )

        text += (

            "\nMODEL PERFORMANCE\n"
            "----------------------------------------\n"

            "Model: NGBoost NGBRegressor\n"

            f"Target: {self.target_name}\n"

            f"Test R²: "
            f"{self.test_r2:.6f}\n"

            f"Test RMSE: "
            f"{self.test_rmse:.6f}\n"

            f"Test MAE: "
            f"{self.test_mae:.6f}\n"

            f"Test MAPE: "
            f"{self.test_mape:.4f}%\n\n"

            "MODEL HYPERPARAMETERS\n"
            "----------------------------------------\n"

            f"n_estimators: "
            f"{self.N_ESTIMATORS}\n"

            f"learning_rate: "
            f"{self.LEARNING_RATE}\n"

            f"minibatch_frac: "
            f"{self.MINIBATCH_FRAC}\n"

            f"col_sample: "
            f"{self.COL_SAMPLE}\n"

            f"natural_gradient: "
            f"{self.NATURAL_GRADIENT}\n"

            f"tol: "
            f"{self.TOL}\n"

            f"random_state: "
            f"{self.RANDOM_STATE}\n"

        )

        with open(
            filename,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                text
            )

        messagebox.showinfo(
            "Saved",
            "Prediction saved successfully."
        )

    # ============================================================
    # COPY
    # ============================================================

    def copy_to_clipboard(self):

        if self.result_var.get() == "READY":

            return

        text = (

            f"Compressive Strength (CS): "
            f"{self.result_var.get()}"

        )

        if self.interval_var.get():

            text += (

                f"\n"
                f"{self.interval_var.get()}"

            )

        self.root.clipboard_clear()

        self.root.clipboard_append(
            text
        )

        self.root.update()

    # ============================================================
    # ANALYSIS FRAME
    # ============================================================

    def clear_analysis_frame(self):

        for widget in (
            self.analysis_frame
            .winfo_children()
        ):

            widget.destroy()

    # ============================================================

    def show_figure(
        self,
        fig
    ):

        self.clear_analysis_frame()

        canvas = FigureCanvasTkAgg(
            fig,
            self.analysis_frame
        )

        canvas.draw()

        canvas.get_tk_widget().pack(
            fill="both",
            expand=True,
            padx=8,
            pady=8
        )

        self.current_plot = fig

    # ============================================================
    # ACTUAL VS PREDICTED
    # ============================================================

    def plot_actual_vs_predicted(self):

        fig = Figure(
            figsize=(10, 7),
            dpi=100,
            facecolor=self.colors["card"]
        )

        ax = fig.add_subplot(
            111
        )

        ax.set_facecolor(
            self.colors["card"]
        )

        ax.scatter(
            self.y_test,
            self.y_test_pred,
            alpha=0.75,
            s=60
        )

        minimum = min(

            self.y_test.min(),

            self.y_test_pred.min()

        )

        maximum = max(

            self.y_test.max(),

            self.y_test_pred.max()

        )

        ax.plot(
            [minimum, maximum],
            [minimum, maximum],
            linestyle="--",
            linewidth=2
        )

        ax.set_xlabel(
            "Actual CS",
            fontsize=11
        )

        ax.set_ylabel(
            "Predicted CS",
            fontsize=11
        )

        ax.set_title(
            "Actual vs Predicted Compressive Strength",
            fontsize=15,
            fontweight="bold"
        )

        ax.grid(
            True,
            alpha=0.2
        )

        text = (

            f"R² = {self.test_r2:.4f}\n"

            f"RMSE = {self.test_rmse:.4f}\n"

            f"MAE = {self.test_mae:.4f}\n"

            f"MAPE = {self.test_mape:.2f}%\n"

            f"n = {len(self.X_test)}"

        )

        ax.text(
            0.04,
            0.96,
            text,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=10,
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor="white",
                alpha=0.9
            )
        )

        fig.tight_layout()

        self.show_figure(
            fig
        )

    # ============================================================
    # RESIDUAL ANALYSIS
    # ============================================================

    def plot_residual_analysis(self):

        residuals = (

            self.y_test.to_numpy()
            -
            self.y_test_pred

        )

        fig = Figure(
            figsize=(10, 7),
            dpi=100,
            facecolor=self.colors["card"]
        )

        ax = fig.add_subplot(
            111
        )

        ax.set_facecolor(
            self.colors["card"]
        )

        ax.scatter(
            self.y_test_pred,
            residuals,
            alpha=0.75,
            s=60
        )

        ax.axhline(
            0,
            linestyle="--",
            linewidth=2
        )

        ax.set_xlabel(
            "Predicted CS",
            fontsize=11
        )

        ax.set_ylabel(
            "Residual (Actual − Predicted)",
            fontsize=11
        )

        ax.set_title(
            "Residual Analysis",
            fontsize=15,
            fontweight="bold"
        )

        ax.grid(
            True,
            alpha=0.2
        )

        fig.tight_layout()

        self.show_figure(
            fig
        )

    # ============================================================
    # RESIDUAL HISTOGRAM
    # ============================================================

    def plot_residual_histogram(self):

        residuals = (

            self.y_test.to_numpy()
            -
            self.y_test_pred

        )

        fig = Figure(
            figsize=(10, 7),
            dpi=100,
            facecolor=self.colors["card"]
        )

        ax = fig.add_subplot(
            111
        )

        ax.set_facecolor(
            self.colors["card"]
        )

        ax.hist(
            residuals,
            bins=20,
            alpha=0.75
        )

        ax.axvline(
            0,
            linestyle="--",
            linewidth=2
        )

        ax.set_xlabel(
            "Residual",
            fontsize=11
        )

        ax.set_ylabel(
            "Frequency",
            fontsize=11
        )

        ax.set_title(
            "Residual Distribution",
            fontsize=15,
            fontweight="bold"
        )

        ax.grid(
            True,
            alpha=0.2
        )

        fig.tight_layout()

        self.show_figure(
            fig
        )

    # ============================================================
    # CORRELATION MATRIX
    # ============================================================

    def plot_correlation_matrix(self):

        numeric_data = (

            self.X_full
            .select_dtypes(
                include=np.number
            )
            .copy()

        )

        numeric_data[
            self.target_name
        ] = self.y_full.values

        if numeric_data.shape[1] < 2:

            messagebox.showinfo(
                "Correlation",
                "Not enough numerical variables."
            )

            return

        corr = (
            numeric_data.corr()
        )

        fig = Figure(
            figsize=(11, 9),
            dpi=100,
            facecolor=self.colors["card"]
        )

        ax = fig.add_subplot(
            111
        )

        ax.set_facecolor(
            self.colors["card"]
        )

        image = ax.imshow(
            corr,
            vmin=-1,
            vmax=1,
            aspect="auto"
        )

        labels = [

            str(x)[:18]

            for x in corr.columns

        ]

        ax.set_xticks(
            np.arange(
                len(labels)
            )
        )

        ax.set_yticks(
            np.arange(
                len(labels)
            )
        )

        ax.set_xticklabels(
            labels,
            rotation=45,
            ha="right",
            fontsize=8
        )

        ax.set_yticklabels(
            labels,
            fontsize=8
        )

        for i in range(
            len(labels)
        ):

            for j in range(
                len(labels)
            ):

                ax.text(
                    j,
                    i,
                    f"{corr.iloc[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7
                )

        fig.colorbar(
            image,
            ax=ax,
            fraction=0.046,
            pad=0.04
        )

        ax.set_title(
            "Correlation Matrix",
            fontsize=15,
            fontweight="bold"
        )

        fig.tight_layout()

        self.show_figure(
            fig
        )

    # ============================================================
    # RUN
    # ============================================================

    def run(self):

        print(
            "\n" + "=" * 75
        )

        print(
            "COMPRESSIVE STRENGTH PREDICTOR"
        )

        print(
            "=" * 75
        )

        print(
            f"Target: "
            f"{self.target_name}"
        )

        print(
            f"Model: "
            f"NGBoost NGBRegressor"
        )

        print(
            f"Features: "
            f"{len(self.feature_names)}"
        )

        print(
            f"Training samples: "
            f"{len(self.X_train)}"
        )

        print(
            f"Test samples: "
            f"{len(self.X_test)}"
        )

        print(
            f"Test R²: "
            f"{self.test_r2:.6f}"
        )

        print(
            f"Test RMSE: "
            f"{self.test_rmse:.6f}"
        )

        print(
            f"Test MAE: "
            f"{self.test_mae:.6f}"
        )

        if np.isfinite(
            self.test_mape
        ):

            print(
                f"Test MAPE: "
                f"{self.test_mape:.4f}%"
            )

        print(
            "=" * 75
        )

        print(
            "GUI READY"
        )

        print(
            "=" * 75
        )

        self.root.mainloop()

    # ============================================================
    # CLOSE
    # ============================================================

    def on_closing(self):

        self.save_history()

        self.root.destroy()


# ================================================================
# MAIN
# ================================================================

if __name__ == "__main__":

    try:

        start_time = time.time()

        print(
            "\nInitializing "
            "Compressive Strength Predictor..."
        )

        app = (
            CompressiveStrengthPredictor()
        )

        print(
            f"\nApplication initialization: "
            f"{time.time() - start_time:.2f} s"
        )

        app.run()

    except Exception as e:

        print(
            "\n" + "=" * 75
        )

        print(
            "APPLICATION ERROR"
        )

        print(
            "=" * 75
        )

        print(
            str(e)
        )

        print(
            "=" * 75
        )

        input(
            "\nPress Enter to exit..."
        )


Initializing Compressive Strength Predictor...

LOADING COMPRESSIVE STRENGTH DATA

APPLICATION ERROR

Dataset not found.

Expected path:
D:\2026 Work\My Papers\waste glass concrete\Data and Main Paper\Preprocessed\_Data\Final\_data_sushant_preprocessed.csv




Press Enter to exit... 
